In [70]:
# ## linking institutions from mag with list of all institutions

import subprocess
import sqlite3 as sqlite
import argparse
import os
import pandas as pd
import numpy as np 
os.chdir("../../")
print(os.getcwd())


from src.dataprep.helpers.functions import *
from src.dataprep.helpers.variables import *

con=sqlite.connect(db_file)





/home


In [71]:
# ## mag sample: Check institutions and names
query="select AffiliationId, NormalizedName, Latitude, Longitude from affiliations where iso3166code = 'US'"
mag=pd.read_sql(sql=query, con=con)
cnt = mag.count
print(cnt)
mag.head()




<bound method DataFrame.count of       AffiliationId                              NormalizedName   Latitude  \
0          50614327                         amec foster wheeler  41.880707   
1          80046288                           walden university  44.981327   
2          94975175                       des moines university  41.584000   
3         158506100  pennsylvania board of probation and parole  40.259860   
4         192633058                  university of saint joseph  41.779600   
...             ...                                         ...        ...   
9212     2802557736                     south seminole hospital  28.699026   
9213     2802694066                st johns river state college  29.646343   
9214     2802841168                    hawaii community college  19.704170   
9215     2802927936                          midland university  41.437420   
9216     2913133470                          high museum of art  33.789240   

       Longitude  
0     -87.6

,AffiliationId,NormalizedName,Latitude,Longitude
0,50614327,amec foster wheeler,41.880707,-87.67417
1,80046288,walden university,44.981327,-93.26623
2,94975175,des moines university,41.584000,-93.66200
3,158506100,pennsylvania board of probation and parole,40.259860,-76.88223
4,192633058,university of saint joseph,41.779600,-72.73050


In [80]:
# ## institutions

query= "select * from institutions"
institutions=pd.read_sql(sql=query, con=con)
del institutions['name']
del institutions['city']

# ## TODO: Decide if Puerto Rico should be included (86 institutions in PR)
institutions= institutions[institutions.stabbr != "PR"]
cnt = institutions.count
print(cnt)
institutions.head()


<bound method DataFrame.count of       unitid stabbr  basic2021  iclevel  \
0     177834     MO         25        1   
1     134811     FL         22        1   
2     429094     TX         26        1   
3     404994     NY         14        1   
4     446127     FL         10        2   
...      ...    ...        ...      ...   
3934  493619     CA         12        2   
3935  141361     GA         21        1   
3936  206695     OH         18        1   
3937  126119     CA          8        2   
3938  204255     OH          6        1   

                                        normalized_name normalized_city  
0               a t still university of health sciences      kirksville  
1     ai miami international university of art and d...           miami  
2          aoma graduate school of integrative medicine          austin  
3                                           asa college        brooklyn  
4                                  ata career education     spring hill  
...   

,unitid,stabbr,basic2021,iclevel,normalized_name,normalized_city
0,177834,MO,25,1,a t still university of health sciences,kirksville
1,134811,FL,22,1,ai miami international university of art and d...,miami
2,429094,TX,26,1,aoma graduate school of integrative medicine,austin
3,404994,NY,14,1,asa college,brooklyn
4,446127,FL,10,2,ata career education,spring hill


In [73]:
# TODO: Do we wanna keep all institutions from mag and institutions? 
#mag_inst=pd.merge(institutions, mag, left_on='normalized_name', right_on='NormalizedName')
#cnt = mag_inst.count
#print(cnt)
#mag_inst.head()
#mag_inst.isnull().any()

In [81]:
# adjusting the name doesn't change anything
excl={"of", "and", "for", "the"}
for char in excl:
    normalized_name=institutions.normalized_name.replace(char, "")
    NormalizedName=mag.NormalizedName.replace(char, "")


In [92]:
# ## merge while keeping only institutions from institution-sample (2154 observations in both samples)
mag_inst2=pd.merge(institutions, mag, left_on='normalized_name', right_on='NormalizedName', how='left', indicator='origin')
cnt = mag_inst2.count
print(cnt)
mag_inst2.head()


<bound method DataFrame.count of       unitid stabbr  basic2021  iclevel  \
0     177834     MO         25        1   
1     134811     FL         22        1   
2     429094     TX         26        1   
3     404994     NY         14        1   
4     446127     FL         10        2   
...      ...    ...        ...      ...   
3870  493619     CA         12        2   
3871  141361     GA         21        1   
3872  206695     OH         18        1   
3873  126119     CA          8        2   
3874  204255     OH          6        1   

                                        normalized_name normalized_city  \
0               a t still university of health sciences      kirksville   
1     ai miami international university of art and d...           miami   
2          aoma graduate school of integrative medicine          austin   
3                                           asa college        brooklyn   
4                                  ata career education     spring hill   


,unitid,stabbr,basic2021,iclevel,normalized_name,normalized_city,AffiliationId,NormalizedName,Latitude,Longitude,origin
0,177834,MO,25,1,a t still university of health sciences,kirksville,NaN,NaN,NaN,NaN,left_only
1,134811,FL,22,1,ai miami international university of art and d...,miami,NaN,NaN,NaN,NaN,left_only
2,429094,TX,26,1,aoma graduate school of integrative medicine,austin,NaN,NaN,NaN,NaN,left_only
3,404994,NY,14,1,asa college,brooklyn,2.802823e+09,asa college,40.6929,-73.9859,both
4,446127,FL,10,2,ata career education,spring hill,NaN,NaN,NaN,NaN,left_only


In [91]:
# Check if merging worked:

print(mag_inst2.isnull().any())

print(mag_inst2['normalized_name'][mag_inst2['NormalizedName'].isnull()].unique())


unitid             False
stabbr             False
basic2021          False
iclevel            False
normalized_name    False
normalized_city    False
AffiliationId       True
NormalizedName      True
Latitude            True
Longitude           True
origin             False
dtype: bool
['a t still university of health sciences'
 'ai miami international university of art and design'
 'aoma graduate school of integrative medicine' ...
 'yo san university of traditional chinese medicine' 'york college'
 'young americans college of the performing arts']
      unitid stabbr  basic2021  iclevel  \
0     177834     MO         25        1   
1     134811     FL         22        1   
2     429094     TX         26        1   
3     404994     NY         14        1   
4     446127     FL         10        2   
...      ...    ...        ...      ...   
3870  493619     CA         12        2   
3871  141361     GA         21        1   
3872  206695     OH         18        1   
3873  126119  

In [85]:
# Summary of new dataset:

both = (mag_inst2['origin'] == 'both').sum()
print(both, 'institutions were included in both datasets.')

left = (mag_inst2['origin'] == 'left_only').sum()
print(left, 'institutions were included only in institutions.')

right = (mag_inst2['origin'] == 'right_only').sum()
print(right, 'institutions were included only in mag.')

2154 institutions were included in both datasets.
1721 institutions were included only in institutions.
0 institutions were included only in mag.


In [95]:
# ## TODO: proquest sample: Check institutions and names
query="select university_id, originalname, normalizedname, location"
pq=pd.read_sql(sql=query, con=con)
cnt = pq.count
print(cnt)
pq.head()

DatabaseError: Execution failed on sql 'select pq_unis': no such column: pq_unis

In [96]:

con.close()

print("Done.")

Done.
